# CIFAR-10: ResNet18 / WideResNet + MixUp/CutMix (50k)

This notebook runs **new experiments** on CIFAR-10 (train split only):
- Model: **ResNet18** (scratch)
- Model: **WideResNet-28-10** (scratch)
- Augmentations: **MixUp** and/or **CutMix** in the `tf.data` pipeline

Results are appended to:
- `artifacts/experiments.jsonl`
- `artifacts/experiments.csv`


In [8]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('artifacts/mplconfig'))

print('TensorFlow:', tf.__version__)
print('Num GPUs Available:', len(tf.config.list_physical_devices('GPU')))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
assert len(class_names) == 10


TensorFlow: 2.20.0
Num GPUs Available: 0


## Experiment config

Recommended default: start with **50 epochs** and 50k samples.


In [9]:
EXP_ID = 'exp09'
EXP_NAME = '50k_resnet18_mixup'
DATASET_NAME = 'CIFAR-10 (train full)'
N_SAMPLES = 50_000

# Split ratios (must sum to 1.0)
SPLIT_TRAIN = 0.80
SPLIT_VAL = 0.10
SPLIT_TEST = 0.10
RANDOM_STATE = 42

# Training budget
# CPU-friendly default
BATCH_SIZE = 50
EPOCHS = 50

# Model choice: 'resnet18' | 'wideresnet28_10'
MODEL_TYPE = 'resnet18'

# Augmentations
USE_FLIP_CROP = True
USE_MIXUP = True
USE_CUTMIX = False
MIXUP_ALPHA = 0.2
CUTMIX_ALPHA = 1.0

# Optimizer
LR = 1e-3
WEIGHT_DECAY = 1e-4


## Load CIFAR-10 (local tar)

Uses the same local loader as `EDA.ipynb`.


In [10]:
import sys
sys.path.insert(0, 'scripts')
from local_cifar10 import load_cifar10_from_tar

# Adjust the path if needed (same as EDA.ipynb setup)
data = load_cifar10_from_tar()
X_train_full, y_train_full = data.x_train, data.y_train
X_test_official, y_test_official = data.x_test, data.y_test
print('train_full:', X_train_full.shape, y_train_full.shape, X_train_full.dtype)
print('test_official:', X_test_official.shape, y_test_official.shape)

# We do NOT use the official CIFAR-10 test set here to avoid leakage with the notebook's internal split.
X = X_train_full[:N_SAMPLES]
y = y_train_full[:N_SAMPLES]
print('subset:', X.shape, y.shape)


train_full: (50000, 32, 32, 3) (50000, 1) uint8
test_official: (10000, 32, 32, 3) (10000, 1)
subset: (50000, 32, 32, 3) (50000, 1)


## Split train/val/test (stratified)


In [11]:
from sklearn.model_selection import train_test_split

y_flat = y.reshape(-1)

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=SPLIT_TEST,
    random_state=RANDOM_STATE,
    stratify=y_flat,
)
val_frac_of_trainval = SPLIT_VAL / (1.0 - SPLIT_TEST)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_frac_of_trainval,
    random_state=RANDOM_STATE,
    stratify=y_trainval.reshape(-1),
)

print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_val  :', X_val.shape, 'y_val  :', y_val.shape)
print('X_test :', X_test.shape, 'y_test :', y_test.shape)


X_train: (40000, 32, 32, 3) y_train: (40000, 1)
X_val  : (5000, 32, 32, 3) y_val  : (5000, 1)
X_test : (5000, 32, 32, 3) y_test : (5000, 1)


## tf.data pipeline + MixUp/CutMix

Images are normalized to `[0, 1]`. Labels are one-hot for MixUp/CutMix.


In [12]:
AUTOTUNE = tf.data.AUTOTUNE

def one_hot(y, num_classes=10):
    # y comes as shape (1,) or (batch, 1); make it (batch,) then one-hot -> (batch, num_classes)
    y = tf.cast(tf.squeeze(y, axis=-1), tf.int32)
    return tf.one_hot(y, depth=num_classes)

def preprocess(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    y = one_hot(y)
    return x, y

def aug_flip_crop(x, y):
    if USE_FLIP_CROP:
        x = tf.image.random_flip_left_right(x)
        x = tf.image.resize_with_crop_or_pad(x, 36, 36)
        x = tf.image.random_crop(x, size=[32, 32, 3])
    return x, y

def sample_beta(alpha, shape):
    g1 = tf.random.gamma(shape, alpha)
    g2 = tf.random.gamma(shape, alpha)
    return g1 / (g1 + g2)

def mixup(batch_x, batch_y, alpha=0.2):
    bs = tf.shape(batch_x)[0]
    lam = sample_beta(alpha, [bs])
    lam_x = tf.reshape(lam, [bs, 1, 1, 1])
    lam_y = tf.reshape(lam, [bs, 1])

    idx = tf.random.shuffle(tf.range(bs))
    x2 = tf.gather(batch_x, idx)
    y2 = tf.gather(batch_y, idx)

    x = batch_x * lam_x + x2 * (1.0 - lam_x)
    y = batch_y * lam_y + y2 * (1.0 - lam_y)
    return x, y

def cutmix(batch_x, batch_y, alpha=1.0):
    # CutMix with a single rectangle per image.
    bs = tf.shape(batch_x)[0]
    lam = sample_beta(alpha, [bs])
    idx = tf.random.shuffle(tf.range(bs))
    x2 = tf.gather(batch_x, idx)
    y2 = tf.gather(batch_y, idx)

    w = tf.cast(tf.shape(batch_x)[2], tf.float32)
    h = tf.cast(tf.shape(batch_x)[1], tf.float32)
    cut_rat = tf.sqrt(1.0 - lam)
    cut_w = tf.cast(w * cut_rat, tf.int32)
    cut_h = tf.cast(h * cut_rat, tf.int32)

    # Uniform center
    cx = tf.random.uniform([bs], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    cy = tf.random.uniform([bs], 0, tf.cast(h, tf.int32), dtype=tf.int32)

    x1 = tf.clip_by_value(cx - cut_w // 2, 0, tf.cast(w, tf.int32))
    x2b = tf.clip_by_value(cx + cut_w // 2, 0, tf.cast(w, tf.int32))
    y1 = tf.clip_by_value(cy - cut_h // 2, 0, tf.cast(h, tf.int32))
    y2b = tf.clip_by_value(cy + cut_h // 2, 0, tf.cast(h, tf.int32))

    def _cutmix_one(args):
        x1i, x2i, y1i, y2i, im1, im2, y1v, y2v = args
        # Replace patch
        pad_top = y1i
        pad_left = x1i
        pad_bottom = tf.shape(im1)[0] - y2i
        pad_right = tf.shape(im1)[1] - x2i
        patch = im2[y1i:y2i, x1i:x2i, :]
        patch = tf.pad(patch, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]])
        mask = tf.ones([y2i - y1i, x2i - x1i, 3], dtype=im1.dtype)
        mask = tf.pad(mask, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]])
        mixed = im1 * (1.0 - mask) + patch

        area = tf.cast((x2i - x1i) * (y2i - y1i), tf.float32)
        lam_adj = 1.0 - area / (w * h)
        y_mixed = y1v * lam_adj + y2v * (1.0 - lam_adj)
        return mixed, y_mixed

    mixed_x, mixed_y = tf.map_fn(
        _cutmix_one,
        (x1, x2b, y1, y2b, batch_x, x2, batch_y, y2),
        fn_output_signature=(tf.float32, tf.float32),
    )
    return mixed_x, mixed_y

def make_ds(X, y, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(50_000, seed=RANDOM_STATE, reshuffle_each_iteration=True)
    ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(aug_flip_crop, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)
    if training and USE_MIXUP:
        ds = ds.map(lambda x, y: mixup(x, y, alpha=MIXUP_ALPHA), num_parallel_calls=AUTOTUNE)
    if training and USE_CUTMIX:
        ds = ds.map(lambda x, y: cutmix(x, y, alpha=CUTMIX_ALPHA), num_parallel_calls=AUTOTUNE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(X_train, y_train, training=True)
val_ds = make_ds(X_val, y_val, training=False)
test_ds = make_ds(X_test, y_test, training=False)

print('MixUp:', USE_MIXUP, 'CutMix:', USE_CUTMIX, 'Flip/Crop:', USE_FLIP_CROP)


MixUp: True CutMix: False Flip/Crop: True


## Models: ResNet18 / WideResNet-28-10 (Keras)

Both models are trained from scratch on CIFAR-10.


In [13]:
def conv3x3(filters, stride=1, wd=1e-4):
    return layers.Conv2D(
        filters, 3, strides=stride, padding='same', use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
    )

def conv1x1(filters, stride=1, wd=1e-4):
    return layers.Conv2D(
        filters, 1, strides=stride, padding='same', use_bias=False,
        kernel_regularizer=tf.keras.regularizers.l2(wd),
    )

def basic_block(x, filters, stride=1, wd=1e-4):
    shortcut = x
    x = conv3x3(filters, stride=stride, wd=wd)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = conv3x3(filters, stride=1, wd=wd)(x)
    x = layers.BatchNormalization()(x)

    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = conv1x1(filters, stride=stride, wd=wd)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x

def build_resnet18(input_shape=(32, 32, 3), num_classes=10, wd=1e-4):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(64, 3, strides=1, padding='same', use_bias=False,
                      kernel_regularizer=tf.keras.regularizers.l2(wd))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # CIFAR-style: no initial maxpool
    x = basic_block(x, 64, stride=1, wd=wd)
    x = basic_block(x, 64, stride=1, wd=wd)

    x = basic_block(x, 128, stride=2, wd=wd)
    x = basic_block(x, 128, stride=1, wd=wd)

    x = basic_block(x, 256, stride=2, wd=wd)
    x = basic_block(x, 256, stride=1, wd=wd)

    x = basic_block(x, 512, stride=2, wd=wd)
    x = basic_block(x, 512, stride=1, wd=wd)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

def wrn_block(x, filters, stride, dropout_rate, wd=5e-4):
    y = layers.BatchNormalization()(x)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False,
                      kernel_regularizer=tf.keras.regularizers.l2(wd))(y)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    if dropout_rate and dropout_rate > 0:
        y = layers.Dropout(dropout_rate)(y)
    y = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False,
                      kernel_regularizer=tf.keras.regularizers.l2(wd))(y)

    if stride != 1 or x.shape[-1] != filters:
        x = layers.Conv2D(filters, 1, strides=stride, padding='same', use_bias=False,
                          kernel_regularizer=tf.keras.regularizers.l2(wd))(x)
    return layers.Add()([x, y])

def build_wideresnet(depth=28, width=10, dropout_rate=0.0, input_shape=(32, 32, 3), num_classes=10, wd=5e-4):
    # WideResNet-28-10: depth = 6n + 4 -> n=(depth-4)/6 = 4
    assert (depth - 4) % 6 == 0
    n = (depth - 4) // 6
    k = width
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(16, 3, strides=1, padding='same', use_bias=False,
                      kernel_regularizer=tf.keras.regularizers.l2(wd))(inputs)

    # group 1
    for i in range(n):
        x = wrn_block(x, 16 * k, stride=1, dropout_rate=dropout_rate, wd=wd)
    # group 2
    x = wrn_block(x, 32 * k, stride=2, dropout_rate=dropout_rate, wd=wd)
    for i in range(n - 1):
        x = wrn_block(x, 32 * k, stride=1, dropout_rate=dropout_rate, wd=wd)
    # group 3
    x = wrn_block(x, 64 * k, stride=2, dropout_rate=dropout_rate, wd=wd)
    for i in range(n - 1):
        x = wrn_block(x, 64 * k, stride=1, dropout_rate=dropout_rate, wd=wd)

    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

if MODEL_TYPE == 'resnet18':
    model = build_resnet18(wd=WEIGHT_DECAY)
elif MODEL_TYPE == 'wideresnet28_10':
    model = build_wideresnet(depth=28, width=10, dropout_rate=0.0)
else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 32, 32,    │      1,728 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_17 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 32, 32,    │     36,864 │ re_lu_17[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_18 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 32, 32,    │     36,864 │ re_lu_18[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ re_lu_17[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_19 (ReLU)     │ (None, 32, 32,    │          0 │ add_8[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │     36,864 │ re_lu_19[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_23[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_20 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_24 (Conv2D)  │ (None, 32, 32,    │     36,864 │ re_lu_20[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_24[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 32, 32,    │          0 │ batch_normalizat

 Total params: 11,183,562 (42.66 MB)

 Trainable params: 11,173,962 (42.63 MB)

 Non-trainable params: 9,600 (37.50 KB)

## Train


In [14]:
loss = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.AdamW(learning_rate=LR, weight_decay=WEIGHT_DECAY)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1211s 2s/step - accuracy: 0.3433 - loss: 2.3626 - val_accuracy: 0.4704 - val_loss: 1.9768 - learning_rate: 0.0010
Epoch 2/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1084s 1s/step - accuracy: 0.5587 - loss: 1.7125 - val_accuracy: 0.5732 - val_loss: 1.6508 - learning_rate: 0.0010
Epoch 3/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1634s 2s/step - accuracy: 0.6325 - loss: 1.4933 - val_accuracy: 0.4800 - val_loss: 1.8003 - learning_rate: 0.0010
Epoch 4/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1477s 2s/step - accuracy: 0.6755 - loss: 1.3912 - val_accuracy: 0.6638 - val_loss: 1.2410 - learning_rate: 0.0010
Epoch 5/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 973s 1s/step - accuracy: 0.6978 - loss: 1.3555 - val_accuracy: 0.3802 - val_loss: 2.8698 - learning_rate: 0.0010
Epoch 6/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 1045s 1s/step - accuracy: 0.7057 - loss: 1.3366 - val_accuracy: 0.6004 - val_loss: 1.4350 - learning_rate: 0.0010
Epoch 7/50
800/800 ━━━━━━━━━━━━━━━━━━━━ 924s 1s/step - accuracy: 0.7324 - los

## Evaluate + log to shared summary


In [15]:
def eval_ds(ds):
    out = model.evaluate(ds, verbose=0, return_dict=True)
    return {'loss': float(out['loss']), 'acc': float(out['accuracy'])}

metrics_train = eval_ds(train_ds)
metrics_val = eval_ds(val_ds)
metrics_test = eval_ds(test_ds)

best_val_acc = float(max(history.history.get('val_accuracy', [float('nan')])))

print('train:', metrics_train)
print('val  :', metrics_val)
print('test :', metrics_test)

from scripts.experiment_log import append_and_export

JSONL_PATH = 'artifacts/experiments.jsonl'
CSV_PATH = 'artifacts/experiments.csv'

aug = []
if USE_FLIP_CROP:
    aug.append('flip_crop')
if USE_MIXUP:
    aug.append(f'mixup(a={MIXUP_ALPHA})')
if USE_CUTMIX:
    aug.append(f'cutmix(a={CUTMIX_ALPHA})')

row = {
    'exp_id': EXP_ID,
    'exp_name': EXP_NAME,
    'dataset': DATASET_NAME,
    'n_samples': int(N_SAMPLES),
    'split_train': float(SPLIT_TRAIN),
    'split_val': float(SPLIT_VAL),
    'split_test': float(SPLIT_TEST),
    'random_state': int(RANDOM_STATE),
    'model_family': 'Scratch CNN + modern aug',
    'model_name': ('ResNet18 (scratch)' if MODEL_TYPE == 'resnet18' else 'WideResNet-28-10 (scratch)'),
    'train_acc': float(metrics_train['acc']),
    'val_acc': float(metrics_val['acc']),
    'test_acc': float(metrics_test['acc']),
    'train_loss': float(metrics_train['loss']),
    'val_loss': float(metrics_val['loss']),
    'test_loss': float(metrics_test['loss']),
    'best_val_acc': best_val_acc,
    'batch_size': int(BATCH_SIZE),
    'epochs': int(EPOCHS),
    'lr': float(LR),
    'weight_decay': float(WEIGHT_DECAY),
    'augmentations': ','.join(aug),
    'notes': '',
}

append_and_export(JSONL_PATH, CSV_PATH, row)
print('Saved to:', JSONL_PATH)
print('Exported:', CSV_PATH)


train: {'loss': 0.5694056749343872, 'acc': 0.9525250196456909}
val  : {'loss': 0.36192548274993896, 'acc': 0.9355999827384949}
test : {'loss': 0.35239529609680176, 'acc': 0.932200014591217}
Saved to: artifacts/experiments.jsonl
Exported: artifacts/experiments.csv
